# NeoBank India — Customer Churn Analysis
## Part 2: Feature Engineering & Exploratory Data Analysis

---

**Objective**

This notebook builds on the cleaned dataset from Part 1. The goals are:
1. Engineer meaningful features from raw columns to enrich the analysis
2. Explore distributions of key variables
3. Identify which features are most strongly associated with churn
4. Derive actionable business insights and recommendations

**Notebook Structure**

| Section | Description |
|---|---|
| 1 | Import Libraries |
| 2 | Load Cleaned Data |
| 3 | Feature Engineering |
| 4 | Univariate Analysis |
| 5 | Churn Analysis — Numerical Features |
| 6 | Churn Analysis — Categorical Features (with Cramér's V effect sizes) |
| 7 | Engineered Feature Analysis |
| 8 | Key Findings & Business Recommendations |

---

**Tech Stack:** Python · pandas · NumPy · Matplotlib · Seaborn · SciPy

---

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

---
## 2. Load Cleaned Dataset

In [ ]:
df = pd.read_csv('neobank_cleaned.csv')

# Fix negative monthly transactions before any analysis
df['monthly_transactions'] = df['monthly_transactions'].abs()

print(f"Dataset shape : {df.shape}")
print(f"Churn rate    : {df['churn'].mean()*100:.1f}%")
df.head()

---
## 3. Feature Engineering

Raw columns alone don't always capture the full picture of a customer's relationship with the bank. We engineer five new features that combine multiple signals into meaningful segments.

| Feature | Based On | Purpose |
|---|---|---|
| `engagement_score` | Transactions, app logins, recency | Measures behavioural activity |
| `customer_status` | engagement_score | Classifies customers as Active, At Risk, or Inactive |
| `tenure_segment` | tenure_months | Groups customers by relationship length |
| `age_group` | age | Groups customers by life stage |
| `balance_segment` | avg_balance | Classifies customers by financial value |
| `credit_risk_category` | credit_score | Classifies customers by credit health |
| `product_profile` | All product columns + financial behaviour | Comprehensive customer relationship type |

### 3.1 Customer Engagement Score & Status

Instead of a single hard rule, we score each customer across three independent behavioural signals — transaction frequency, app login frequency, and transaction recency. Each condition met adds 1 point (max score = 3).

- **Score 3** → Active — engaged across all dimensions
- **Score 2** → At Risk — partially disengaged, showing early warning signs
- **Score 0-1** → Inactive — disengaged across most signals

The At Risk segment is the most actionable for retention — these customers can still be saved.

In [ ]:
df['engagement_score'] = 0

# Thresholds are set at the approximate median of each behavioural metric across the dataset:
# - monthly_transactions >= 8  (median: ~8 transactions/month)
# - app_logins_per_month >= 8  (median: ~8 logins/month)
# - days_since_last_transaction <= 15  (median recency cutoff)
# A customer meeting all three signals is considered genuinely active.

# Signal 1: Transaction frequency (at or above median)
df['engagement_score'] += (df['monthly_transactions'] >= 8).astype(int)

# Signal 2: App login frequency (at or above median)
df['engagement_score'] += (df['app_logins_per_month'] >= 8).astype(int)

# Signal 3: Transaction recency (transacted within last 15 days)
df['engagement_score'] += (df['days_since_last_transaction'] <= 15).astype(int)

def classify_customer(score):
    if score == 3:
        return 'Active'
    elif score == 2:
        return 'At Risk'
    else:
        return 'Inactive'

df['customer_status'] = df['engagement_score'].apply(classify_customer)

print("Customer Status Distribution:")
print(df['customer_status'].value_counts())
print(f"\nActive   : {df['customer_status'].eq('Active').mean()*100:.1f}%")
print(f"At Risk  : {df['customer_status'].eq('At Risk').mean()*100:.1f}%")
print(f"Inactive : {df['customer_status'].eq('Inactive').mean()*100:.1f}%")

### 3.2 Tenure Segment

Groups customers by how long they have been with the bank. Longer tenure generally indicates a more stable and loyal customer.

In [ ]:
def tenure_segment(months):
    if months <= 6:
        return 'New'
    elif months <= 24:
        return 'Developing'
    elif months <= 60:
        return 'Established'
    else:
        return 'Loyal'

df['tenure_segment'] = df['tenure_months'].apply(tenure_segment)

print("Tenure Segment Distribution:")
print(df['tenure_segment'].value_counts())

### 3.3 Age Group

Groups customers by life stage. Different age groups have different banking needs, digital engagement habits, and churn behaviour.

In [ ]:
def age_group(age):
    if age <= 25:
        return 'Young Adult'
    elif age <= 40:
        return 'Mid Age'
    elif age <= 60:
        return 'Senior'
    else:
        return 'Elderly'

df['age_group'] = df['age'].apply(age_group)

print("Age Group Distribution:")
print(df['age_group'].value_counts())

### 3.4 Balance Segment

Classifies customers by average balance into four value tiers. Higher value customers are generally more invested in the bank relationship.

In [ ]:
def balance_segment(balance):
    if balance < 10000:
        return 'Low Value'
    elif balance < 50000:
        return 'Mid Value'
    elif balance < 200000:
        return 'High Value'
    else:
        return 'Premium'

df['balance_segment'] = df['avg_balance'].apply(balance_segment)

print("Balance Segment Distribution:")
print(df['balance_segment'].value_counts())

### 3.5 Credit Risk Category

Classifies customers based on credit score ranges used by Indian credit bureaus (CIBIL standard).

In [ ]:
def credit_risk(score):
    if score >= 750:
        return 'Excellent'
    elif score >= 650:
        return 'Good'
    elif score >= 550:
        return 'Fair'
    else:
        return 'Poor'

df['credit_risk_category'] = df['credit_score'].apply(credit_risk)

print("Credit Risk Category Distribution:")
print(df['credit_risk_category'].value_counts())

### 3.6 Product Profile

The most comprehensive engineered feature. Combines product holdings with financial behaviour to classify customers into 9 meaningful relationship types.

| Profile | Description | Expected Churn |
|---|---|---|
| Premium Relationship | High value, diversified, financially stable | Very Low |
| Stable Homeowner | Home loan holder, reliable payer | Very Low |
| Wealth Builder | Investment focused, no debt pressure | Low |
| Responsible Credit User | Salaried, stable credit, insured | Low |
| Stressed Borrower | Has loans but struggling to repay | Medium |
| Bonus Hunter | Credit card only, low balance | High |
| Financially Stressed | Poor credit, low balance, minimal products | High |
| Minimal Relationship | Only one basic product, no depth | High |
| Standard Customer | Average customer, no strong signal | Medium |

In [ ]:
def product_profile(row):
    has_cc     = row['has_credit_card']   == 'Yes'
    has_pl     = row['has_personal_loan'] == 'Yes'
    has_hl     = row['has_home_loan']     == 'Yes'
    has_fd     = row['has_fixed_deposit'] == 'Yes'
    has_demat  = row['has_demat_account'] == 'Yes'
    has_ins    = row['has_insurance']     == 'Yes'

    has_loan       = has_pl or has_hl
    has_investment = has_fd or has_demat
    good_credit    = row['credit_score'] >= 700
    low_credit     = row['credit_score'] < 550
    high_balance   = row['avg_balance']  >= 50000
    low_balance    = row['avg_balance']  < 10000
    late_payer     = row['late_payments'] >= 2

    if has_loan and has_investment and has_ins and good_credit and high_balance:
        return 'Premium Relationship'
    if has_hl and good_credit and not late_payer:
        return 'Stable Homeowner'
    if has_investment and not has_loan and high_balance:
        return 'Wealth Builder'
    if has_cc and has_ins and good_credit and not late_payer:
        return 'Responsible Credit User'
    if has_loan and (late_payer or low_credit):
        return 'Stressed Borrower'
    if has_cc and not has_loan and not has_investment and low_balance:
        return 'Bonus Hunter'
    if low_credit and low_balance and row['num_products'] <= 2:
        return 'Financially Stressed'
    if row['num_products'] <= 1:
        return 'Minimal Relationship'
    return 'Standard Customer'

df['product_profile'] = df.apply(product_profile, axis=1)

print("Product Profile Distribution:")
print(df['product_profile'].value_counts())

In [ ]:
print(f"Final dataset shape after feature engineering: {df.shape}")
print(f"\nNew columns added:")
new_cols = ['engagement_score', 'customer_status', 'tenure_segment',
            'age_group', 'balance_segment', 'credit_risk_category', 'product_profile']
print(new_cols)

---
## 4. Univariate Analysis

We examine the distribution of key numerical columns to understand the customer base before diving into churn analysis.

### 4.1 Age Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['age'], bins=30, edgecolor='black', color='steelblue')
axes[0].set_title('Age — Histogram')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

sns.kdeplot(df['age'], fill=True, ax=axes[1], color='steelblue')
axes[1].set_title('Age — Density Plot')
axes[1].set_xlabel('Age')

sns.boxplot(x=df['age'], ax=axes[2], color='steelblue')
axes[2].set_title('Age — Boxplot')

plt.suptitle('Age Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Mean age   : {df['age'].mean():.1f}")
print(f"Median age : {df['age'].median():.1f}")
print(f"Age range  : {df['age'].min()} to {df['age'].max()}")

**Observation:** The age distribution is right-skewed with the majority of customers between 25-50 years — the core working-age population. The boxplot confirms no extreme outliers after cleaning.

### 4.2 Tenure Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['tenure_months'], bins=30, edgecolor='black', color='coral')
axes[0].set_title('Tenure — Histogram')
axes[0].set_xlabel('Tenure (Months)')
axes[0].set_ylabel('Count')

sns.kdeplot(df['tenure_months'], fill=True, ax=axes[1], color='coral')
axes[1].set_title('Tenure — Density Plot')
axes[1].set_xlabel('Tenure (Months)')

sns.boxplot(x=df['tenure_months'], ax=axes[2], color='coral')
axes[2].set_title('Tenure — Boxplot')

plt.suptitle('Tenure Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Mean tenure   : {df['tenure_months'].mean():.1f} months")
print(f"Median tenure : {df['tenure_months'].median():.1f} months")
print(f"Tenure range  : {df['tenure_months'].min()} to {df['tenure_months'].max()} months")

**Observation:** Tenure is right-skewed — the bank has a large base of newer customers (consistent with recent aggressive digital acquisition) and a smaller but significant loyal base with 60+ months tenure.

### 4.3 Average Balance Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['avg_balance'], bins=40, edgecolor='black', color='mediumseagreen')
axes[0].set_title('Avg Balance — Histogram')
axes[0].set_xlabel('Average Balance (₹)')
axes[0].set_ylabel('Count')

sns.kdeplot(df['avg_balance'], fill=True, ax=axes[1], color='mediumseagreen')
axes[1].set_title('Avg Balance — Density Plot')
axes[1].set_xlabel('Average Balance (₹)')

sns.boxplot(x=df['avg_balance'], ax=axes[2], color='mediumseagreen')
axes[2].set_title('Avg Balance — Boxplot')

plt.suptitle('Average Balance Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Mean balance   : ₹{df['avg_balance'].mean():,.0f}")
print(f"Median balance : ₹{df['avg_balance'].median():,.0f}")
print(f"Balance range  : ₹{df['avg_balance'].min():,.0f} to ₹{df['avg_balance'].max():,.0f}")

**Observation:** Balance is heavily right-skewed — most customers maintain moderate balances while a small number hold very high balances. The mean is significantly higher than the median, indicating the high-balance tail is pulling the average up.

### 4.4 Credit Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['credit_score'], bins=30, edgecolor='black', color='mediumpurple')
axes[0].set_title('Credit Score — Histogram')
axes[0].set_xlabel('Credit Score')
axes[0].set_ylabel('Count')

sns.kdeplot(df['credit_score'], fill=True, ax=axes[1], color='mediumpurple')
axes[1].set_title('Credit Score — Density Plot')
axes[1].set_xlabel('Credit Score')

sns.boxplot(x=df['credit_score'], ax=axes[2], color='mediumpurple')
axes[2].set_title('Credit Score — Boxplot')

plt.suptitle('Credit Score Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Mean credit score   : {df['credit_score'].mean():.0f}")
print(f"Median credit score : {df['credit_score'].median():.0f}")
print(f"Score range         : {df['credit_score'].min():.0f} to {df['credit_score'].max():.0f}")

**Observation:** Credit scores follow an approximately normal distribution centred around 650-700, which aligns with the typical Indian banking customer profile.

---
## 5. Churn Analysis — Numerical Features

### 5.1 Correlation With Churn

We calculate the correlation coefficient between each numerical feature and the churn variable to identify which features have the strongest linear relationship with churn.

In [ ]:
churn_corr = df.corr(numeric_only=True)['churn'].drop('churn').sort_values(ascending=False)
print("Correlation with Churn:")
print(churn_corr.round(4))

In [ ]:
plt.figure(figsize=(10, 6))

colors = ['coral' if x > 0 else 'steelblue' for x in churn_corr.values]
plt.barh(churn_corr.index, churn_corr.values, color=colors, edgecolor='black')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.title('Correlation of Features with Churn', fontsize=14, fontweight='bold')
plt.xlabel('Correlation Coefficient', fontsize=12)

for i, value in enumerate(churn_corr.values):
    plt.text(value + 0.01 if value >= 0 else value - 0.01,
             i, f'{value:.3f}',
             va='center',
             ha='left' if value >= 0 else 'right',
             fontsize=9)

plt.tight_layout()
plt.show()

**Observation:** Two features stand out clearly:
- `days_since_last_transaction` (0.87) — strongest positive correlation. Customers who have not transacted recently are highly likely to churn.
- `engagement_score` (-0.49) — strongest negative correlation. Higher engagement strongly predicts retention.

Notably, financial indicators like `credit_score`, `avg_balance`, and `late_payments` show near-zero correlation with churn. This suggests **churn in this bank is driven by behavioural disengagement, not financial stress.**

### 5.2 Correlation Heatmap — All Numerical Features

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5
)
plt.title('Correlation Heatmap — All Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Observation:** `days_since_last_transaction` and `engagement_score` show strong opposite correlations with churn — consistent with the bar chart above. Most other numerical features show weak inter-correlations, confirming they capture independent signals.

### 5.3 Key Behavioural Signals vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='churn', y='days_since_last_transaction',
            data=df, ax=axes[0], palette={0: 'steelblue', 1: 'coral'})
axes[0].set_title('Days Since Last Transaction vs Churn', fontsize=13)
axes[0].set_xlabel('Churn (0 = No, 1 = Yes)', fontsize=11)
axes[0].set_ylabel('Days Since Last Transaction', fontsize=11)

sns.boxplot(x='churn', y='engagement_score',
            data=df, ax=axes[1], palette={0: 'steelblue', 1: 'coral'})
axes[1].set_title('Engagement Score vs Churn', fontsize=13)
axes[1].set_xlabel('Churn (0 = No, 1 = Yes)', fontsize=11)
axes[1].set_ylabel('Engagement Score', fontsize=11)

plt.suptitle('Behavioural Signals vs Churn', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Observation:** Churned customers (1) show dramatically higher days since last transaction and much lower engagement scores compared to retained customers (0). The separation is clear and consistent — confirming these are the two most reliable churn indicators in this dataset.

---
## 6. Churn Analysis — Categorical Features

For categorical features, we use churn rate by category rather than correlation. We also run chi-square tests to verify statistical significance.

### 6.1 Statistical Significance — Chi-Square Test

A p-value below 0.05 means the relationship between that feature and churn is statistically significant and not due to random chance.

In [ ]:
def chi_square_test(column):
    contingency_table = pd.crosstab(df[column], df['churn'])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    significant = 'Yes ✅' if p_value < 0.05 else 'No ❌'
    print(f"{column:<25} Chi2: {chi2:>8.2f}   P-value: {p_value:.4f}   Significant: {significant}")

print(f"{'Feature':<25} {'Chi-Square':>14}   {'P-Value':>10}   Significant")
print("-" * 70)
for col in ['customer_status', 'product_profile', 'occupation',
            'balance_segment', 'city_tier', 'tenure_segment',
            'age_group', 'credit_risk_category', 'account_type']:
    chi_square_test(col)

### 6.1b Effect Size — Cramér's V

A statistically significant chi-square result tells us the relationship exists, but not how strong it is. **Cramér's V** measures practical significance on a 0–1 scale (0 = no association, 1 = perfect association). This gives a clearer picture of which categorical features actually matter.

Rule of thumb: V < 0.1 = negligible, 0.1–0.3 = small, 0.3–0.5 = medium, > 0.5 = large.

In [ ]:
def cramers_v(column):
    contingency_table = pd.crosstab(df[column], df['churn'])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    phi2 = chi2 / n
    r, k = contingency_table.shape
    v = np.sqrt(phi2 / min(k - 1, r - 1))
    significance = 'Yes' if p_value < 0.05 else 'No'
    return v, p_value, significance

print(f"{'Feature':<25} {'Cramér\'s V':>12}   {'P-Value':>10}   {'Significant':>12}   {'Effect Size':>12}")
print("-" * 80)
cols = ['customer_status', 'product_profile', 'occupation',
        'balance_segment', 'city_tier', 'tenure_segment',
        'age_group', 'credit_risk_category', 'account_type']

for col in cols:
    v, p, sig = cramers_v(col)
    if v >= 0.5:
        effect = 'Large'
    elif v >= 0.3:
        effect = 'Medium'
    elif v >= 0.1:
        effect = 'Small'
    else:
        effect = 'Negligible'
    print(f"{col:<25} {v:>12.4f}   {p:>10.4f}   {sig:>12}   {effect:>12}")

print("\n> Note: account_type's negligible Cramér's V confirms it has no meaningful")
print("> relationship with churn — this is a substantive finding, not a sample-size artefact.")

**Observation:** `customer_status`, `product_profile`, `occupation`, `balance_segment`, and `city_tier` are all statistically significant predictors of churn. `account_type` is the only feature that does not reach significance.

> **Statistical note:** Chi-square significance depends on sample size — large datasets can flag trivially small differences as significant. Cramér's V (computed below) provides the effect size to confirm practical importance alongside statistical significance.

### 6.2 Churn Rate by City Tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

city_counts = df['city_tier'].value_counts()
churn_by_tier = df.groupby('city_tier')['churn'].mean() * 100

axes[0].bar(city_counts.index, city_counts.values, color='steelblue', edgecolor='black')
axes[0].set_title('Customer Count by City Tier', fontsize=13)
axes[0].set_xlabel('City Tier', fontsize=11)
axes[0].set_ylabel('Number of Customers', fontsize=11)
for i, v in enumerate(city_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=10)

axes[1].bar(churn_by_tier.index, churn_by_tier.values, color='coral', edgecolor='black')
axes[1].set_title('Churn Rate by City Tier (%)', fontsize=13)
axes[1].set_xlabel('City Tier', fontsize=11)
axes[1].set_ylabel('Churn Rate (%)', fontsize=11)
for i, v in enumerate(churn_by_tier.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)

plt.suptitle('City Tier — Distribution & Churn Rate', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Observation:** Metro has both the highest customer count and the highest churn rate — suggesting the bank is acquiring customers in its largest market but struggling to retain them, possibly due to greater competition from other banks.

### 6.3 Churn Rate by Occupation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

occ_counts = df['occupation'].value_counts()
churn_by_occ = df.groupby('occupation')['churn'].mean() * 100

axes[0].bar(occ_counts.index, occ_counts.values, color='steelblue', edgecolor='black')
axes[0].set_title('Customer Count by Occupation', fontsize=13)
axes[0].set_xlabel('Occupation', fontsize=11)
axes[0].set_ylabel('Number of Customers', fontsize=11)
for i, v in enumerate(occ_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=10)

axes[1].bar(churn_by_occ.index, churn_by_occ.values, color='coral', edgecolor='black')
axes[1].set_title('Churn Rate by Occupation (%)', fontsize=13)
axes[1].set_xlabel('Occupation', fontsize=11)
axes[1].set_ylabel('Churn Rate (%)', fontsize=11)
for i, v in enumerate(churn_by_occ.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)

plt.suptitle('Occupation — Distribution & Churn Rate', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Observation:** Students show the highest churn rate — consistent with their transient lifestyle and lower financial commitment to the bank. Retired customers show the lowest churn — they are the most stable and loyal segment.

---
## 7. Engineered Feature Analysis

### 7.1 Churn Rate by Engineered Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Customer Status
status_churn = df.groupby('customer_status')['churn'].mean() * 100
status_churn = status_churn.sort_values(ascending=True)
axes[0].barh(status_churn.index, status_churn.values, color='steelblue', edgecolor='black')
axes[0].set_title('Churn Rate by Customer Status', fontsize=12)
axes[0].set_xlabel('Churn Rate (%)')
for i, v in enumerate(status_churn.values):
    axes[0].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)

# Balance Segment
balance_churn = df.groupby('balance_segment')['churn'].mean() * 100
balance_churn = balance_churn.sort_values(ascending=True)
axes[1].barh(balance_churn.index, balance_churn.values, color='coral', edgecolor='black')
axes[1].set_title('Churn Rate by Balance Segment', fontsize=12)
axes[1].set_xlabel('Churn Rate (%)')
for i, v in enumerate(balance_churn.values):
    axes[1].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)

# Product Profile
profile_churn = df.groupby('product_profile')['churn'].mean() * 100
profile_churn = profile_churn.sort_values(ascending=True)
axes[2].barh(profile_churn.index, profile_churn.values, color='mediumseagreen', edgecolor='black')
axes[2].set_title('Churn Rate by Product Profile', fontsize=12)
axes[2].set_xlabel('Churn Rate (%)')
for i, v in enumerate(profile_churn.values):
    axes[2].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)

plt.suptitle('Churn Rate by Engineered Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Observation:** The engineered features reveal stark differences in churn rates:
- Inactive customers churn at 59% vs only 0.8% for Active customers — a 74x difference
- Low Value customers churn at more than double the rate of Premium customers
- Financially Stressed and Minimal Relationship profiles show the highest churn while Stable Homeowners show the lowest

### 7.2 Product Profile vs Customer Status

Cross-tabulation showing what proportion of each product profile falls into each customer status category.

In [ ]:
crosstab = pd.crosstab(df['product_profile'], df['customer_status'], normalize='index') * 100
crosstab = crosstab.round(1)
print("Product Profile vs Customer Status (% within each profile):")
print(crosstab)

**Key observations from this table:**
- **Financially Stressed** has the highest At Risk rate (49.3%) — nearly half are showing disengagement signals
- **Wealth Builder** has the second highest At Risk rate (38.5%) — investment customers may be disengaging after their FD matured
- **Stressed Borrowers** show high Active rate (61.5%) — but this is obligation-driven activity from loan repayments, not genuine engagement
- **Stable Homeowners** show the highest Active rate (62.6%) and lowest Inactive rate (4.1%)

### 7.3 Validation — Stressed Borrower Activity Pattern

In [ ]:
print("Stressed Borrower — Monthly Transaction Statistics:")
stressed = df[df['product_profile'] == 'Stressed Borrower']
print(stressed['monthly_transactions'].describe())

print(f"\nMean transactions (Stressed Borrower) : {stressed['monthly_transactions'].mean():.1f}")
print(f"Mean transactions (all customers)     : {df['monthly_transactions'].mean():.1f}")
print("\nConclusion: Stressed Borrowers transact more than average, but this is")
print("likely driven by loan EMI payments — obligation, not loyalty.")

---
## 8. Key Findings & Business Recommendations

---

### Key Findings

**Finding 1 — Churn is behavioural, not financial**

`days_since_last_transaction` shows a 0.87 correlation with churn — the strongest predictor in the dataset. Financial indicators like credit score (-0.04) and average balance (-0.04) show near-zero correlation. Customers are not leaving because they are financially stressed — they are leaving because they are behaviourally disengaging.

---

**Finding 2 — Engagement score is a powerful early warning signal**

The composite engagement score shows -0.49 correlation with churn. Inactive customers (engagement score 0-1) churn at 59% while Active customers churn at only 0.8% — a 74x difference. Monitoring engagement score provides an early warning before a customer formally churns.

---

**Finding 3 — Product depth is a strong retention driver**

`num_products` shows -0.20 correlation with churn. Stable Homeowners (home loan + other products) churn at only 1.74% while Minimal Relationship customers (1 product) churn at 21.07%. Customers with deeper product relationships are significantly harder to lose.

---

**Finding 4 — Financially Stressed and Minimal Relationship are highest risk segments**

Financially Stressed customers churn at 31.34% and Minimal Relationship customers at 21.07% — the two highest churn segments. Both have low product depth and low balance, giving them little reason to stay.

---

**Finding 5 — Stressed Borrowers are active but not loyal**

Stressed Borrowers show high transaction activity (mean 23.5/month) but this is driven by loan EMI obligations rather than genuine engagement. Once their loan is repaid, this segment loses its primary reason to stay.

---

**Finding 6 — Metro is the highest churn market**

Metro has both the highest customer count and the highest churn rate — suggesting strong acquisition but weak retention in the most competitive market.

---

### Business Recommendations

**Recommendation 1 — Build a behavioural early warning system**

Monitor `engagement_score` and `days_since_last_transaction` weekly for every customer. Flag customers whose engagement score drops below 2 or who haven't transacted in 30+ days for proactive outreach. Intervening before behavioural disengagement becomes formal churn is the highest-leverage retention action available.

---

**Recommendation 2 — Targeted re-engagement for Inactive customers**

Inactive customers churn at 59%. A dedicated re-engagement campaign — personalised offers, fee waivers, or cashback on first transaction — specifically targeting customers with `customer_status = Inactive` could meaningfully reduce overall churn rate.

---

**Recommendation 3 — Cross-sell to single product customers**

Minimal Relationship customers (1 product, 21% churn) are the easiest to cross-sell to since they have the most to gain. Introducing a second product — insurance or a fixed deposit — at the 3-6 month tenure mark could significantly increase stickiness before early disengagement sets in.

---

**Recommendation 4 — Develop a post-loan retention strategy for Stressed Borrowers**

Stressed Borrowers are currently retained by obligation. The bank should proactively introduce them to other products (insurance, savings plans) during the loan tenure so that when the loan ends, there is another reason to stay.

---

**Recommendation 5 — Prioritise retention spend in Metro**

Metro shows the highest churn rate despite being the largest customer segment. Targeted loyalty programmes or preferential rates for Metro customers could reduce churn in the bank's most important market.

---

*Next steps: Dashboard development in Tableau/Power BI to visualise churn segments for the retention team.*

---
*See `README.md` for project overview and setup instructions.*